# Valoración de una opción *call* americana con dividendos y volatilidad con cambio de régimen (Markov)
## Monte Carlo, Longstaff–Schwartz e Importance Sampling (IS)

Este notebook está preparado como **pipeline reproducible** para el TFG:

1. **Modelo**: subyacente con *dividend yield* continuo \(q\) y volatilidad que alterna entre dos regímenes (baja/alta) según una cadena de Markov.
2. **Europea**: estimación por Monte Carlo estándar y por **Importance Sampling** (reducción de varianza) con control de sesgo mediante pesos.
3. **Americana**: valoración por **Longstaff–Schwartz (LSM)**, aprendiendo una política de ejercicio (tiempo de ejercicio por trayectoria).
4. **Americana + IS**: valoración de la **misma política** bajo IS (mismo estimador, menor varianza), separando *policy learning* y *policy valuation*.

> Nota metodológica: en opciones americanas, IS debe usarse con cuidado. La forma más robusta (y defendible) es **aprender la política con simulación estándar** y **valorarla con IS**. Así evitamos inestabilidades de regresión con pesos.

Todo el código está comentado en español y pensado para ser citables/explicables en memoria.


In [1]:
# ========== 0) Configuración y librerías ==========
import numpy as np
import pandas as pd
import time
from dataclasses import dataclass
from math import erf, sqrt, log, exp

# Reproducibilidad global (semillas)
GLOBAL_SEED = 20260205
np.random.seed(GLOBAL_SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


## 1) Parámetros base del experimento

Estos parámetros sirven como **baseline** para pruebas y para construir tablas en el Capítulo de resultados.
Puedes modificar `Ks`, `T_list`, `n_paths` y `n_steps` para tus escenarios (ITM/ATM/OTM y vencimientos).


In [2]:
# ========== 1) Parámetros base ==========
S0 = 100.0

# Tipos (continuos)
r = 0.03      # tipo libre de riesgo
q = 0.02      # dividend yield continuo (en el TFG es clave: incentiva ejercicio anticipado)

# Regímenes de volatilidad
sigma_L = 0.18
sigma_H = 0.40

# Matriz de transición Markov (2 estados)
P = np.array([[0.95, 0.05],
              [0.10, 0.90]], dtype=float)

# Discretización temporal
T_list = [1.0]          # puedes añadir [0.25, 1.0, 2.0] en resultados
n_steps = 252           # 252 días de mercado (aprox.)

# Tamaño Monte Carlo
n_paths = 50_000        # para resultados finos puedes subir a 100k-200k (tiempo ↑)

# Strikes (moneyness)
Ks = [90.0, 100.0, 110.0, 120.0]  # añadir 120 ayuda a ver el efecto de IS en cola OTM

print("Parámetros cargados.")


Parámetros cargados.


## 2) Simulación del modelo

Bajo medida neutral al riesgo:
\[
dS_t = (r-q)S_t\,dt + \sigma(X_t)S_t\,dW_t
\]
donde \(X_t\in\{0,1\}\) es una cadena de Markov discreta (baja/alta volatilidad).

Discretización por paso (exacta para GBM con volatilidad constante en el paso):
\[
S_{t+\Delta t} = S_t\exp\Big((r-q-\tfrac12\sigma^2)\Delta t + \sigma\sqrt{\Delta t}\,Z\Big)
\]


In [3]:
# ========== 2) Estructuras de parámetros ==========
@dataclass(frozen=True)
class RegimeParams:
    sigma: np.ndarray  # [sigma_L, sigma_H]
    P: np.ndarray      # 2x2 transición

@dataclass(frozen=True)
class MarketParams:
    r: float
    q: float

def validate_params(regime: RegimeParams, market: MarketParams):
    sigma = np.asarray(regime.sigma, dtype=float)
    Pm = np.asarray(regime.P, dtype=float)

    if sigma.shape != (2,) or np.any(sigma <= 0):
        raise ValueError("sigma debe tener shape (2,) y valores positivos.")
    if Pm.shape != (2,2) or np.any(Pm < 0) or not np.allclose(Pm.sum(axis=1), 1.0, atol=1e-12):
        raise ValueError("P debe ser 2x2, no negativa, y filas deben sumar 1.")
    if not (-0.05 <= market.r <= 0.20):
        raise ValueError("r fuera de rango razonable.")
    if not (0.0 <= market.q <= 0.30):
        raise ValueError("q fuera de rango razonable (>=0).")

def simulate_markov_regimes(P: np.ndarray, n_paths: int, n_steps: int, x0: int = 0, seed: int = 0):
    """Simula cadena de Markov binaria {0,1}. Devuelve X (n_paths, n_steps+1)."""
    rng = np.random.default_rng(seed)
    X = np.empty((n_paths, n_steps+1), dtype=np.int8)
    X[:, 0] = x0

    U = rng.random((n_paths, n_steps))
    p_to_0 = P[:, 0]  # umbral para ir a 0 desde cada estado

    for t in range(n_steps):
        prev = X[:, t]
        thr = p_to_0[prev]
        X[:, t+1] = (U[:, t] >= thr).astype(np.int8)

    return X

def simulate_paths_standard(S0: float, T: float, n_steps: int, n_paths: int,
                            market: MarketParams, regime: RegimeParams,
                            x0: int = 0, seed: int = 0):
    """Simulación estándar (sin IS). Devuelve t_grid, S, X."""
    validate_params(regime, market)
    dt = T / n_steps
    t_grid = np.linspace(0.0, T, n_steps+1)

    X = simulate_markov_regimes(regime.P, n_paths, n_steps, x0=x0, seed=seed)
    rng = np.random.default_rng(seed + 1)
    Z = rng.standard_normal((n_paths, n_steps))

    S = np.empty((n_paths, n_steps+1), dtype=float)
    S[:, 0] = S0

    sigma_states = np.asarray(regime.sigma, dtype=float)
    for t in range(n_steps):
        sig = sigma_states[X[:, t]]
        drift = (market.r - market.q - 0.5 * sig**2) * dt
        diff = sig * np.sqrt(dt) * Z[:, t]
        S[:, t+1] = S[:, t] * np.exp(drift + diff)

    return t_grid, S, X

# Instanciamos parámetros
market = MarketParams(r=r, q=q)
regime = RegimeParams(sigma=np.array([sigma_L, sigma_H], dtype=float), P=P)

print("Funciones de simulación listas.")


Funciones de simulación listas.


## 3) Valoración europea (MC estándar) + Black–Scholes (caso constante)

La call europea bajo MC se estima como:
\[
C^{Eu} = e^{-rT}\,\mathbb{E}[(S_T-K)^+]
\]
y su error estándar se obtiene de la desviación muestral del pay-off descontado.

Además, para validar el motor, incluimos la fórmula de **Black–Scholes con dividend yield** (solo aplica si \(\sigma\) es constante).


In [4]:
# ========== 3) Europea MC y Black–Scholes ==========
def price_european_call_mc(S_T: np.ndarray, K: float, r: float, T: float):
    payoff = np.maximum(S_T - K, 0.0)
    disc = np.exp(-r * T)
    contrib = disc * payoff
    price = contrib.mean()
    se = contrib.std(ddof=1) / np.sqrt(contrib.size)
    return price, se, price - 1.96*se, price + 1.96*se

def norm_cdf(x: float) -> float:
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def black_scholes_call(S0: float, K: float, r: float, q: float, sigma: float, T: float) -> float:
    if T <= 0:
        return max(S0 - K, 0.0)
    if sigma <= 0:
        ST = S0 * exp((r - q) * T)
        return exp(-r*T) * max(ST - K, 0.0)
    d1 = (log(S0 / K) + (r - q + 0.5*sigma*sigma)*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    return S0*exp(-q*T)*norm_cdf(d1) - K*exp(-r*T)*norm_cdf(d2)

print("Funciones de pricing europeo listas.")


Funciones de pricing europeo listas.


## 4) Importance Sampling (IS) para europea

Usamos un *tilting* simple y robusto:
\[
Z \sim \mathcal{N}(0,1) \quad\to\quad Z^{(\lambda)} = Z + \lambda
\]
El peso (likelihood ratio) por trayectoria es:
\[
W = \exp\left(-\lambda\sum_{k=1}^N Z_k - \tfrac12\lambda^2 N\right)
\]
y el estimador insesgado:
\[
\hat{C}^{IS} = e^{-rT}\,\frac{1}{M}\sum_{i=1}^M W^{(i)}(S_T^{(i)}-K)^+
\]

> Para americana necesitaremos **pesos acumulados** por fecha \(W_t\). Aquí ya los generamos.


In [5]:
# ========== 4) Simulación con IS (pesos acumulados) ==========
def stationary_dist_2state(P):
    p01 = P[0,1]
    p10 = P[1,0]
    pi0 = p10 / (p01 + p10)
    pi1 = 1.0 - pi0
    return np.array([pi0, pi1])

def choose_lambda_proxy(S0, K_target, r, q, T, sigma_L, sigma_H, P):
    """Heurística: desplaza la media de log S_T hacia log(K_target) usando una volatilidad media."""
    pi = stationary_dist_2state(P)
    sig_bar = np.sqrt(pi[0]*sigma_L**2 + pi[1]*sigma_H**2)
    m0 = np.log(S0) + (r - q - 0.5*sig_bar**2)*T
    target = np.log(K_target)
    lam = (target - m0) / (sig_bar * np.sqrt(T))
    return float(lam)

def simulate_paths_IS_with_cumweights(S0: float, T: float, n_steps: int, n_paths: int,
                                     market: MarketParams, regime: RegimeParams,
                                     lam: float, x0: int = 0, seed: int = 0):
    """Simulación con IS: Z -> Z + lam. Devuelve t_grid, S, X y pesos acumulados W(t)."""
    validate_params(regime, market)
    dt = T / n_steps
    t_grid = np.linspace(0.0, T, n_steps+1)

    X = simulate_markov_regimes(regime.P, n_paths, n_steps, x0=x0, seed=seed)

    rng = np.random.default_rng(seed + 1)
    Z = rng.standard_normal((n_paths, n_steps))
    Z_tilt = Z + lam

    S = np.empty((n_paths, n_steps+1), dtype=float)
    S[:, 0] = S0

    # log-weights acumulados: logW_{t+1} = logW_t + (-lam*Z - 0.5 lam^2)
    logW = np.zeros((n_paths, n_steps+1), dtype=float)
    logw_step = -lam * Z - 0.5 * (lam**2)

    sigma_states = np.asarray(regime.sigma, dtype=float)

    for t in range(n_steps):
        sig = sigma_states[X[:, t]]
        drift = (market.r - market.q - 0.5 * sig**2) * dt
        diff = sig * np.sqrt(dt) * Z_tilt[:, t]
        S[:, t+1] = S[:, t] * np.exp(drift + diff)

        logW[:, t+1] = logW[:, t] + logw_step[:, t]

    W = np.exp(logW)
    return t_grid, S, X, W

def price_european_call_mc_IS(S_T: np.ndarray, W_T: np.ndarray, K: float, r: float, T: float):
    payoff = np.maximum(S_T - K, 0.0)
    contrib = np.exp(-r*T) * W_T * payoff
    price = contrib.mean()
    se = contrib.std(ddof=1) / np.sqrt(contrib.size)
    return price, se, price - 1.96*se, price + 1.96*se

print("IS listo.")


IS listo.


## 5) Longstaff–Schwartz (LSM) para call americana (cashflow-based)

Implementación estándar y robusta:

- Aprendemos el **valor de continuación** \(C(S_t, X_t)\) por regresión en un conjunto de bases (polinomios).
- Decidimos ejercicio cuando:
\[
\text{payoff inmediato} \ge \hat C
\]
- Guardamos por trayectoria el **tiempo de ejercicio** \(\tau_i\) y el **cashflow** \(CF_i\).
- El precio se estima como:
\[
V_0 = \frac{1}{M}\sum_{i=1}^M e^{-r\tau_i\Delta t} CF_i
\]

**Detalle importante**: para el caso **call sin dividendos** (\(q=0\)) se sabe que no hay incentivo al ejercicio anticipado. En el notebook lo usamos como *sanity check* (no como “atajo” del resultado principal, que usa \(q>0\)).


In [6]:
# ========== 5) LSM (cashflow-based) con regresión por régimen ==========
@dataclass(frozen=True)
class LSMPolicy:
    tau_idx: np.ndarray     # índice de ejercicio por path (0..N)
    CF: np.ndarray          # cashflow por path
    pv0: np.ndarray         # valor presente por path
    exercise_rate: float    # % paths con ejercicio anticipado
    mean_tau: float         # tiempo medio (en años) de ejercicio

def _make_basis(x: np.ndarray, degree: int = 2) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    cols = [np.ones_like(x)]
    for d in range(1, degree + 1):
        cols.append(x ** d)
    return np.column_stack(cols)

def lsm_american_call_policy(
    S: np.ndarray,
    X: np.ndarray,
    K: float,
    r: float,
    T: float,
    q_for_theory: float,
    basis_degree: int = 2,
    by_regime: bool = True,
    scale_by_strike: bool = True,
    min_reg_points: int = 500,
    enforce_no_early_if_q0: bool = True
) -> LSMPolicy:
    """
    Devuelve política LSM (tau_idx, CF) para call americana.

    - Si enforce_no_early_if_q0=True y q_for_theory==0: se fuerza tau=N (ejercicio al vencimiento),
      para cumplir la propiedad teórica y evitar falsos positivos por ruido de regresión en el sanity check.
    """
    S = np.asarray(S, float)
    X = np.asarray(X)
    M, Np1 = S.shape
    N = Np1 - 1
    dt = T / N

    payoff = np.maximum(S - K, 0.0)

    # Política por defecto: ejercer en vencimiento
    tau = np.full(M, N, dtype=int)
    CF = payoff[:, -1].copy()

    if enforce_no_early_if_q0 and abs(q_for_theory) < 1e-14:
        pv0 = CF * np.exp(-r * tau * dt)
        return LSMPolicy(tau_idx=tau, CF=CF, pv0=pv0,
                         exercise_rate=0.0, mean_tau=float(T))

    # Backward induction
    for t in range(N - 1, 0, -1):
        alive = (tau > t)  # paths que todavía pueden adelantar ejercicio a t
        if not np.any(alive):
            break

        payoff_t = payoff[alive, t]
        itm = payoff_t > 0
        if not np.any(itm):
            continue

        # Target: PV en t del cashflow actual programado en tau
        tau_alive = tau[alive][itm]
        CF_alive = CF[alive][itm]
        Y = CF_alive * np.exp(-r * (tau_alive - t) * dt)

        S_t = S[alive, t][itm]
        X_t = X[alive, t][itm]
        payoff_itm = payoff_t[itm]

        x_feat = (S_t / K) if scale_by_strike else S_t
        C_hat = np.zeros_like(Y)

        if by_regime:
            for state in (0, 1):
                mask = (X_t == state)
                if mask.sum() < min_reg_points:
                    if mask.sum() > 0:
                        C_hat[mask] = Y[mask].mean()
                    continue
                A = _make_basis(x_feat[mask], degree=basis_degree)
                beta, *_ = np.linalg.lstsq(A, Y[mask], rcond=None)
                C_hat[mask] = A @ beta
        else:
            if x_feat.size >= min_reg_points:
                A = _make_basis(x_feat, degree=basis_degree)
                beta, *_ = np.linalg.lstsq(A, Y, rcond=None)
                C_hat = A @ beta
            else:
                C_hat[:] = Y.mean()

        exercise = payoff_itm >= C_hat

        alive_idx = np.flatnonzero(alive)
        itm_global = alive_idx[np.flatnonzero(itm)]
        ex_global = itm_global[exercise]

        tau[ex_global] = t
        CF[ex_global] = payoff[ex_global, t]

    pv0 = CF * np.exp(-r * tau * dt)
    exercise_rate = float(np.mean(tau < N))
    mean_tau = float(np.mean(tau) * dt)

    return LSMPolicy(tau_idx=tau, CF=CF, pv0=pv0,
                     exercise_rate=exercise_rate, mean_tau=mean_tau)

def price_from_policy(policy: LSMPolicy):
    price = policy.pv0.mean()
    se = policy.pv0.std(ddof=1) / np.sqrt(policy.pv0.size)
    return price, se, price - 1.96*se, price + 1.96*se

print("LSM listo.")


LSM listo.


## 6) Americana + IS: valoración de la política bajo pesos

Con IS, el peso acumulado \(W_t\) depende del tiempo. Si la política decide ejercicio en \(\tau_i\), el estimador IS es:

\[
\hat V^{IS} = \frac{1}{M}\sum_{i=1}^M e^{-r\tau_i\Delta t}\,W_{\tau_i}^{(i)}\,(S_{\tau_i}^{(i)}-K)^+
\]

Aquí **no re-entrenamos** la política con IS (eso requeriría regresión ponderada y puede ser inestable).  
Aprendemos la política con simulación estándar y la valoramos con IS.


In [7]:
# ========== 6) Valorar una política bajo IS ==========
def value_policy_under_IS(S: np.ndarray, W: np.ndarray, tau_idx: np.ndarray, K: float, r: float, T: float):
    S = np.asarray(S, float)
    W = np.asarray(W, float)
    tau_idx = np.asarray(tau_idx, int)

    M, Np1 = S.shape
    N = Np1 - 1
    dt = T / N

    idx = np.arange(M)
    payoff_tau = np.maximum(S[idx, tau_idx] - K, 0.0)
    weight_tau = W[idx, tau_idx]
    disc_tau = np.exp(-r * tau_idx * dt)

    contrib = disc_tau * weight_tau * payoff_tau
    price = contrib.mean()
    se = contrib.std(ddof=1) / np.sqrt(M)
    return price, se, price - 1.96*se, price + 1.96*se

print("Valoración IS de política lista.")


Valoración IS de política lista.


## 7) Runner: ejecutar tablas (Europea estándar vs Europea+IS, y Americana LSM vs Americana+IS)

Este bloque genera una tabla con:
- Precio + SE (y CI95)
- Tiempo de cómputo
- Métrica de eficiencia: \(SE^2 \times tiempo\)

Además, incluye un **sanity check** recomendado:
- Para \(q=0\), una call americana debe coincidir con la europea.


In [8]:
# ========== 7) Runner de experimentos ==========
def run_experiment(T: float, Ks, n_paths: int, n_steps: int,
                   S0: float, r: float, q: float, regime: RegimeParams, P: np.ndarray,
                   sigma_L: float, sigma_H: float,
                   seed_std: int = 123, seed_is: int = 456,
                   lam_target_K: float = 110.0):
    market = MarketParams(r=r, q=q)

    # --- Simulación estándar (para europea + policy LSM)
    t0 = time.perf_counter()
    _, S_std, X_std = simulate_paths_standard(S0, T, n_steps, n_paths, market, regime, x0=0, seed=seed_std)
    time_std_sim = time.perf_counter() - t0

    # --- Lambda para IS (elige objetivo en cola / K alto para notar reducción de varianza)
    lam = choose_lambda_proxy(S0, lam_target_K, r, q, T, sigma_L, sigma_H, P)

    # --- Simulación IS
    t1 = time.perf_counter()
    _, S_is, X_is, W = simulate_paths_IS_with_cumweights(S0, T, n_steps, n_paths, market, regime, lam, x0=0, seed=seed_is)
    time_is_sim = time.perf_counter() - t1

    rows = []
    for K in Ks:
        # Europea estándar
        t2 = time.perf_counter()
        eu_p, eu_se, eu_lo, eu_hi = price_european_call_mc(S_std[:, -1], K, r, T)
        t_eu = time.perf_counter() - t2

        # Europea IS
        t3 = time.perf_counter()
        eu_is_p, eu_is_se, eu_is_lo, eu_is_hi = price_european_call_mc_IS(S_is[:, -1], W[:, -1], K, r, T)
        t_eu_is = time.perf_counter() - t3

        # Americana (policy LSM)
        t4 = time.perf_counter()
        policy = lsm_american_call_policy(
            S_std, X_std, K=K, r=r, T=T, q_for_theory=q,
            basis_degree=2, by_regime=True, scale_by_strike=True,
            min_reg_points=500, enforce_no_early_if_q0=True
        )
        am_p, am_se, am_lo, am_hi = price_from_policy(policy)
        t_am = time.perf_counter() - t4

        # Americana IS (valorar la policy con IS)
        t5 = time.perf_counter()
        am_is_p, am_is_se, am_is_lo, am_is_hi = value_policy_under_IS(S_is, W, policy.tau_idx, K=K, r=r, T=T)
        t_am_is = time.perf_counter() - t5

        rows.append({
            "T": T, "K": K, "lambda": lam,
            "EU_price": eu_p, "EU_SE": eu_se, "EU_CI_low": eu_lo, "EU_CI_high": eu_hi, "EU_time": t_eu,
            "EU_IS_price": eu_is_p, "EU_IS_SE": eu_is_se, "EU_IS_CI_low": eu_is_lo, "EU_IS_CI_high": eu_is_hi, "EU_IS_time": t_eu_is,
            "AM_LSM_price": am_p, "AM_LSM_SE": am_se, "AM_LSM_CI_low": am_lo, "AM_LSM_CI_high": am_hi, "AM_LSM_time": t_am,
            "AM_IS_price": am_is_p, "AM_IS_SE": am_is_se, "AM_IS_CI_low": am_is_lo, "AM_IS_CI_high": am_is_hi, "AM_IS_time": t_am_is,
            "exercise_rate": policy.exercise_rate, "mean_tau": policy.mean_tau,
        })

    df = pd.DataFrame(rows)

    # Eficiencia (menor es mejor)
    df["EU_eff"] = (df["EU_SE"]**2) * (df["EU_time"] + time_std_sim)  # incluimos el coste base de sim en estándar
    df["EU_IS_eff"] = (df["EU_IS_SE"]**2) * (df["EU_IS_time"] + time_is_sim)

    df["AM_eff"] = (df["AM_LSM_SE"]**2) * (df["AM_LSM_time"] + time_std_sim)
    df["AM_IS_eff"] = (df["AM_IS_SE"]**2) * (df["AM_IS_time"] + time_is_sim)

    df["EU_SE_ratio(IS/STD)"] = df["EU_IS_SE"] / df["EU_SE"]
    df["AM_SE_ratio(IS/LSM)"] = df["AM_IS_SE"] / df["AM_LSM_SE"]

    return df

print("Runner listo.")


Runner listo.


In [9]:
# ========== 8) Ejecutar baseline (T=1) ==========
results = []
for T_ in T_list:
    dfT = run_experiment(
        T=T_, Ks=Ks, n_paths=n_paths, n_steps=n_steps,
        S0=S0, r=r, q=q,
        regime=regime, P=P, sigma_L=sigma_L, sigma_H=sigma_H,
        seed_std=111, seed_is=222, lam_target_K=110.0
    )
    results.append(dfT)

df = pd.concat(results, ignore_index=True)
df


,T,K,lambda,EU_price,EU_SE,EU_CI_low,EU_CI_high,EU_time,EU_IS_price,EU_IS_SE,EU_IS_CI_low,EU_IS_CI_high,EU_IS_time,AM_LSM_price,AM_LSM_SE,AM_LSM_CI_low,AM_LSM_CI_high,AM_LSM_time,AM_IS_price,AM_IS_SE,AM_IS_CI_low,AM_IS_CI_high,AM_IS_time,exercise_rate,mean_tau,EU_eff,EU_IS_eff,AM_eff,AM_IS_eff,EU_SE_ratio(IS/STD),AM_SE_ratio(IS/LSM)
0,1.0,90.0,0.448517,15.947575,0.095547,15.760303,16.134846,0.000740,0.382706,0.172605,0.044400,0.721011,0.000740,15.851538,0.073040,15.708379,15.994697,1.154783,2.040095,0.397112,1.261756,2.818434,0.003469,0.36348,0.832577,0.004533,0.024452,0.008806,0.129858,1.806494,5.436876
1,1.0,100.0,0.448517,10.778509,0.082181,10.617434,10.939584,0.000407,0.353902,0.159875,0.040546,0.667257,0.000741,10.775092,0.067723,10.642354,10.907829,1.104936,0.935463,0.288631,0.369747,1.501180,0.004041,0.23282,0.917676,0.003351,0.020978,0.007342,0.068649,1.945403,4.261917
2,1.0,110.0,0.448517,7.023588,0.068395,6.889533,7.157643,0.000464,0.325098,0.147161,0.036663,0.613533,0.000671,7.017637,0.057304,6.905321,7.129953,0.777931,0.361108,0.140400,0.085923,0.636292,0.002348,0.16080,0.951401,0.002321,0.017773,0.004182,0.016210,2.151620,2.450096
3,1.0,120.0,0.448517,4.440993,0.055479,4.332253,4.549732,0.000399,0.296294,0.134465,0.032742,0.559846,0.000625,4.400265,0.046521,4.309083,4.491447,0.556609,0.343856,0.139425,0.070582,0.617130,0.001884,0.10144,0.972260,0.001527,0.014838,0.002278,0.015977,2.423704,2.997017


## 9) Sanity check recomendado: \(q=0\) ⇒ Americana = Europea (call)

Este test no se usa para “calcular el precio final” del caso con dividendos.  
Sirve para comprobar que, cuando no hay dividendos, el algoritmo **no ejerce antes por ruido**.

> Para este test mantenemos el mismo modelo de regímenes y solo cambiamos \(q=0\).


In [10]:
# ========== 9) Sanity check con q=0 ==========
market_q0 = MarketParams(r=r, q=0.0)

_, S_q0, X_q0 = simulate_paths_standard(S0, 1.0, n_steps, n_paths, market_q0, regime, x0=0, seed=333)

rows = []
for K in [90.0, 100.0, 110.0]:
    eu_p, eu_se, eu_lo, eu_hi = price_european_call_mc(S_q0[:, -1], K, r, 1.0)

    policy_q0 = lsm_american_call_policy(
        S_q0, X_q0, K=K, r=r, T=1.0, q_for_theory=0.0,
        basis_degree=2, by_regime=True, scale_by_strike=True,
        min_reg_points=500, enforce_no_early_if_q0=True
    )
    am_p, am_se, am_lo, am_hi = price_from_policy(policy_q0)

    rows.append({
        "K": K,
        "EU": eu_p, "EU_SE": eu_se,
        "AM": am_p, "AM_SE": am_se,
        "AM_minus_EU": am_p - eu_p,
        "exercise_rate": policy_q0.exercise_rate
    })

pd.DataFrame(rows)


,K,EU,EU_SE,AM,AM_SE,AM_minus_EU,exercise_rate
0,90.0,17.743702,0.101547,17.743702,0.101547,0.0,0.0
1,100.0,12.238125,0.088560,12.238125,0.088560,0.0,0.0
2,110.0,8.165536,0.074750,8.165536,0.074750,0.0,0.0


## 10) Salida lista para memoria: tabla resumida (para copiar/pegar)

Recomendación: en el capítulo de resultados, reporta para cada escenario:
- precios y SE
- ratio de SE (IS vs estándar)
- eficiencia \(SE^2\times tiempo\)
- tasa de ejercicio anticipado y tiempo medio de ejercicio (interpretación económica)


In [11]:
# ========== 10) Resumen compacto para el TFG ==========
cols = [
    "T","K","lambda",
    "EU_price","EU_SE","EU_IS_price","EU_IS_SE","EU_SE_ratio(IS/STD)",
    "AM_LSM_price","AM_LSM_SE","AM_IS_price","AM_IS_SE","AM_SE_ratio(IS/LSM)",
    "exercise_rate","mean_tau",
    "EU_eff","EU_IS_eff","AM_eff","AM_IS_eff"
]
df[cols].sort_values(["T","K"]).reset_index(drop=True)


,T,K,lambda,EU_price,EU_SE,EU_IS_price,EU_IS_SE,EU_SE_ratio(IS/STD),AM_LSM_price,AM_LSM_SE,AM_IS_price,AM_IS_SE,AM_SE_ratio(IS/LSM),exercise_rate,mean_tau,EU_eff,EU_IS_eff,AM_eff,AM_IS_eff
0,1.0,90.0,0.448517,15.947575,0.095547,0.382706,0.172605,1.806494,15.851538,0.073040,2.040095,0.397112,5.436876,0.36348,0.832577,0.004533,0.024452,0.008806,0.129858
1,1.0,100.0,0.448517,10.778509,0.082181,0.353902,0.159875,1.945403,10.775092,0.067723,0.935463,0.288631,4.261917,0.23282,0.917676,0.003351,0.020978,0.007342,0.068649
2,1.0,110.0,0.448517,7.023588,0.068395,0.325098,0.147161,2.151620,7.017637,0.057304,0.361108,0.140400,2.450096,0.16080,0.951401,0.002321,0.017773,0.004182,0.016210
3,1.0,120.0,0.448517,4.440993,0.055479,0.296294,0.134465,2.423704,4.400265,0.046521,0.343856,0.139425,2.997017,0.10144,0.972260,0.001527,0.014838,0.002278,0.015977


In [12]:
df.to_csv("pricing_results.csv", index=False)